# Power_vs_Spacing

Y Axis: Spacings

X Axis: Speed.

Feed Rate: (Determined from previous experiments)

Purpose: Inspect half toning / darkness of each set of lines.

In [38]:
import gcode
import numpy as np

In [39]:
def init(**kwargs) -> gcode.GCode:
    prog = gcode.GCode(**kwargs)
    prog.G21()
    prog.G90()
    # Zero on the K30 cross hairs.
    prog.G92(X=0, Y=0)
    # Set motion velocity.
    prog.G1(F=1000)
    prog.G0(F=1000)

    prog.M3(S=0)
    # Cross hairs offset.
    prog.G1(X=16.5, Y=1)
    prog.G92(X=0, Y=0)

    # Offset for axes labels
    prog.G1(X=10, Y=10)
    prog.G92(X=0, Y=0)

    prog.M5()
    return prog

Draw the Axis

In [40]:
Xf = 100
Yf = 40


def add_axes():
    # X Axis
    prog = gcode.GCode()
    prog += gcode.Line(
        points=gcode.hline(X0=0, Xf=Xf, Y=0), power=100, feed=750, dynamic_power=False
    )
    prog += gcode.Line(
        points=gcode.vline(Y0=0, Yf=Yf, X=0), power=100, feed=750, dynamic_power=False
    )
    # Offset to start drawing rectangles.
    prog.G1(X=5, Y=5)
    # Re-zero for the experiment
    prog.G92(X=0, Y=0)

    return prog

In [41]:
powers = np.linspace(0, 100, 101)
powers

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100.])

In [42]:
spacings = [0.1, 0.2, 0.5, 0.75, 1]

In [43]:
prog = init()
prog += add_axes()
speed = 500
spacing = 5

box_width = 100
box_height = 5

Y = 0
X = 0
prog.G1(X=0, Y=0, Z=0, F=speed)
for spacing_idx, spacing in enumerate(spacings):
    for y_idx, y in enumerate(np.arange(0, box_height + spacing, spacing)):
        if np.mod(y_idx, 1) == 2:
            powers_tmp = powers[::-1]
        else:
            powers_tmp = powers
        for power_idx, power in enumerate(powers_tmp):
            prog.M3(S=power)
            prog.G1(X=power, Y=Y + y)
    Y += box_height
    Y += 5

prog.M5()
prog.G1(X=-10, Y=-10, F=2000)
prog.G92(X=0, Y=0)

prog.save("/tmp/Power_vs_Spacing.gcode")